In [ ]:
from openai import OpenAI
import base64
import json
import os

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_image(image_path, client, prompt):
    img_b64_str = encode_image(image_path)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_b64_str}"},
                    },
                ],
            }
        ],
    )
    return response.choices[0].message.content

def process_images(image_list, client, prompt):
    results = {}
    for idx, image_path in enumerate(image_list, start=1):
        result = analyze_image(image_path, client, prompt)
        image_name = os.path.basename(image_path)
        results[idx] = {"image_name": image_name, "content": result}
    return results

def save_results_to_json(results, output_file):
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=4)

if __name__ == "__main__":
    image_list = [
        "examples/code.png",
        "examples/kim.png",
        "examples/chrome.png",
        "examples/mode.png",
        "examples/defillama.png",
        "examples/web_shopping.png",
        "examples/web_forum.png",
        "examples/safari_google.png",
        "examples/app_store.png",
        "examples/ocr.png",
    ]
    output_json = "gpt_4o_results.json"
    client = OpenAI(api_key="")
    prompt = "Please analyze and describe the content of the image"
    results = process_images(image_list, client, prompt)
    save_results_to_json(results, output_json)



In [ ]:
from openai import OpenAI
import json

client = OpenAI(api_key="", base_url="https://api.deepseek.com")

def analyze_content(content, client):
    messages = [
        {
            "role": "user",
            "content": "Please analyze the following information: " + content,
        }
    ]
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=messages
    )
    return response.choices[0].message.content, response.choices[0].message.reasoning_content

def process_json_content(json_data, client):
    results = {}
    for key, value in json_data.items():
        content = value["content"]
        image_name = value["image_name"]
        model_content, reasoning_content = analyze_content(content, client)
        results[key] = {
            "image_name": image_name,
            "gpt_4o_content": content,
            "ds_content": model_content,
            "ds_reasoning_content": reasoning_content
        }
    return results

if __name__ == "__main__":
    input_json = output_json
    ds_output_json = "ds_results.json"

    with open(input_json, 'r') as f:
        json_data = json.load(f)

    results = process_json_content(json_data, client)
    save_results_to_json(results, ds_output_json)

